# Fitting the DC3 NGC 4151 EC1000 Spectrum with a Cutoff Power Law

## Introduction

This notebook demonstrates an unbinned spectral analysis of the DC3 NGC 4151 EC1000 simulation using a cutoff power-law (CPL) spectrum. Only the source-event file is from DC3; the background events, orientation, neural-network response, and neural-network background model are the DC4 files used by `NGC4151_fit_unbinned.ipynb`.

> **Note on Underlying Mechanics:** To keep this tutorial concise, the detailed explanations of the underlying classes (such as `NFResponse`, `NFBackground`, and `UnbinnedThreeMLPointSourceResponseIRFAdaptive`) have been omitted here. If you are unfamiliar with the Neural-Network Response, the Background Approximation, or the corresponding hardware requirements introduced for DC4, **please refer to the foundational tutorial:** `example_grb_fit_normalizing_flows.ipynb`.

## Example

### Basic Setup

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
from pathlib import Path
from cosipy.spacecraftfile import SpacecraftHistory

import astropy.units as u
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np

from threeML import PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Cutoff_powerlaw
from cosipy.util import fetch_wasabi_file

from cosipy.threeml.unbinned_model_folding import CachedUnbinnedThreeMLModelFolding
from cosipy.statistics import UnbinnedLikelihood
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.interfaces.expectation_interface import SumExpectationDensity

from cosipy.event_selection.time_selection import TimeSelector
from cosipy.data_io.EmCDSUnbinnedData import TimeTagEmCDSEventDataInSCFrameFromDC3Fits

from cosipy.response.ml.NFResponse import NFResponse
from cosipy.response.ml.nf_instrument_response_function import UnpolarizedNFFarFieldInstrumentResponseFunction

from cosipy.background_estimation.ml.NFBackground import NFBackground
from cosipy.background_estimation.ml.nf_unbinned_background import FreeNormNFUnbinnedBackground

from cosipy.threeml.ml.optimized_unbinned_folding import UnbinnedThreeMLPointSourceResponseIRFAdaptive

In [ ]:
dc3_path = Path("/home/parshap/COSI/Radio_Quiet_AGN/DC3")
dc4_path = Path("/home/parshap/COSI/Radio_Quiet_AGN/DC4")
pointing_cut_path = dc4_path / "FOV_Cut"
cache_path = dc3_path / "unbinned_cache_EC1000_full"
dc3_path.mkdir(parents=True, exist_ok=True)

ngc4151_data_path = dc3_path / (
    "NGC_4151_EC1000_3months_unbinned_data_filtered_with_SAAcut.fits.gz"
)
if not ngc4151_data_path.exists():
    fetch_wasabi_file(
        "COSI-SMEX/DC3/Data/Sources/"
        "NGC_4151_EC1000_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
        output=ngc4151_data_path,
    )

bkg_data_path = dc4_path / (
    "Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits"
)
sc_orientation_path = dc4_path / (
    "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
)

rsp_path = dc4_path / "unpolarized_nfresponse_v1-01.pt"
bkg_path = dc4_path / "nfbackground_v1-01.pt"

Use the full time range of the pointing-cut orientation file.

In [ ]:
sc_orientation = SpacecraftHistory.open(sc_orientation_path)
tstart = sc_orientation.tstart
tstop = sc_orientation.tstop

print(f"Observation interval: {tstart.isot} to {tstop.isot}")

In [ ]:
data_file = [ngc4151_data_path, bkg_data_path]
selector = TimeSelector(tstart = sc_orientation.tstart, tstop = sc_orientation.tstop)
data = TimeTagEmCDSEventDataInSCFrameFromDC3Fits(data_file, selection=selector)

In [ ]:
print(f"This analysis uses {data.nevents} events")

### Initializing Models and Folding Objects
Setting up the neural-network response (`NFResponse`), the background approximation (`NFBackground`), and the adaptive folding objects.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. Run this notebook inside a GPU Slurm job."
    )

device_name = os.environ.get("COSI_DEVICE", "cuda:0")
device = torch.device(device_name)
if device.type != "cuda" or device.index is None:
    raise ValueError("COSI_DEVICE must be an indexed CUDA device, such as cuda:0")
if device.index >= torch.cuda.device_count():
    raise RuntimeError(
        f"Requested {device_name}, but only {torch.cuda.device_count()} GPU(s) are visible"
    )

devices = [device_name]
print(f"Visible GPUs: {torch.cuda.device_count()}")
print(f"Using {device_name}: {torch.cuda.get_device_name(device)}")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}")

In [ ]:
rsp = NFResponse(
    path_to_model=rsp_path,
    area_batch_size=300_000,
    density_batch_size=100_000, 
    devices=devices,
    area_compile_mode=None,
    density_compile_mode=None,
    show_progress=True)

irf = UnpolarizedNFFarFieldInstrumentResponseFunction(rsp)

In [ ]:
bkg_model = NFBackground(
    path_to_model=bkg_path,
    density_batch_size=100_000,
    devices=devices,
    density_compile_mode=None,
    show_progress=True)

bkg = FreeNormNFUnbinnedBackground(
    model=bkg_model, 
    data=data, 
    sc_history=sc_orientation, 
    label="bkg_norm")

In [ ]:
psr = UnbinnedThreeMLPointSourceResponseIRFAdaptive(
    data=data, 
    irf=irf, 
    sc_history=sc_orientation, 
    show_progress=True, 
    force_energy_node_caching=True, 
    reduce_memory=True) 

psr.cache_batch_size = 100_000
psr.integration_batch_size = 100_000

The DC3 EC1000 source was generated with a single `Cutoff_powerlaw`: normalization 0.15 keV$^{-1}$ s$^{-1}$ cm$^{-2}$ at 1 keV, photon index -1.75, and cutoff energy 1000 keV. All three parameters remain free in the fit. The numerical pivot is moved to 100 keV while preserving the injected spectrum exactly.

In [ ]:
l = 155.07
b = 75.06

index = -1.75  # Injected photon index
piv = 100. * u.keV  # Pivot inside the fitted band for better conditioning
xc = 1000. * u.keV
K_1keV = 0.15 / u.cm / u.cm / u.s / u.keV
K = K_1keV * (piv / (1. * u.keV))**index

spectrum = Cutoff_powerlaw(
    index=index,
    K=K.value,
    piv=piv.value,
    xc=xc.value,
)

spectrum.index.min_value = -5
spectrum.index.max_value = 0

spectrum.xc.min_value = 50
spectrum.xc.max_value = 5000

spectrum.K.min_value = K.value / 100
spectrum.K.max_value = K.value * 100

spectrum.K.unit = K.unit
spectrum.piv.unit = piv.unit
spectrum.xc.unit = xc.unit

spectrum.index.delta = 0.01
spectrum.K.delta = K.value * 0.1
spectrum.xc.delta = 50.

In [ ]:
# Exact spectral form used to generate the simulated source events.
spectrum_inj = deepcopy(spectrum)

In [ ]:
source = PointSource("NGC_4151",
                     l=l,
                     b=b,
                     spectral_shape=spectrum)
model = Model(source)

In [ ]:
response = CachedUnbinnedThreeMLModelFolding(psr)

In [ ]:
expectation_density = SumExpectationDensity(response, bkg)

In [ ]:
like_fun = UnbinnedLikelihood(expectation_density)
cosi = ThreeMLPluginInterface('cosi', like_fun, response, bkg)

In [ ]:
bkg_norm = bkg.norm.to_value(u.Hz)

cosi.bkg_parameter['bkg_norm'] = Parameter("bkg_norm",
                                          bkg_norm,
                                          unit = u.Hz,
                                          min_value=0,
                                          max_value=150,
                                          delta=0.05,
                                          )

print(f"The average background flux is {bkg_norm:.2f} Hz")

In [ ]:
plugins = DataList(cosi)
like = JointLikelihood(model, plugins, verbose=False) # You can enable debugging

### Initializing the Cache

Load the dedicated DC3 EC1000 response cache when available. This cache must not be shared with the DC4 source analysis because the event list is different.

In [ ]:
cache_file = cache_path / "NGC_4151_source_response_cache.h5"
if cache_file.exists():
    response.load_caches(cache_path)
    print(f"Loaded response cache: {cache_file}")
else:
    print(f"No existing cache at {cache_file}; it will be computed.")

The cache is initialized, which takes some time

In [ ]:
print(f"Data Events: {data.nevents}\nExpected Events: {expectation_density.expected_counts():.2f}\nRelative Deviation {100 * (expectation_density.expected_counts()/data.nevents - 1):.3f} %")

Save the newly computed DC3 EC1000 response cache on Palmetto. Existing caches are left untouched.

In [ ]:
if not cache_file.exists():
    response.save_caches(cache_path)
    print(f"Saved response cache: {cache_file}")

### Fitting

In [ ]:
# print("Starting fit...", flush=True)
# _ = like.fit()
# print("Fit complete.", flush=True)

# like.results.display()

# results = like.results

# parameters = {par.name: results.get_variates(par.path)
#               for par in results.optimized_model["NGC_4151"].parameters.values()
#               if par.free}

# results_err = results.propagate(results.optimized_model["NGC_4151"].spectrum.main.shape.evaluate_at, **parameters)

# energy = np.geomspace(100*u.keV, 10*u.MeV).to_value(u.keV)

# flux_lo = np.zeros_like(energy)
# flux_median = np.zeros_like(energy)
# flux_hi = np.zeros_like(energy)
# flux_inj = np.zeros_like(energy)

# for i, e in enumerate(energy):
#     flux = results_err(e)
#     flux_median[i] = flux.median
#     flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)
#     flux_inj[i] = spectrum_inj.evaluate_at(e)
    
# %matplotlib inline

# fig, ax = plt.subplots(figsize = (9, 6))

# ax.plot(energy, flux_median, label = "Best fit")
# ax.fill_between(energy, flux_lo, flux_hi, alpha = .5, label = "Best fit (errors)")
# ax.plot(energy, flux_inj, color = 'black', ls = ":", label = "Injected")

# ax.semilogx()
# ax.semilogy()

# ax.set_xlabel("Energy [keV]")
# ax.set_ylabel(r"$\frac{\mathrm{d}N}{\mathrm{d}E}$ [keV$^{-1}$ cm$^{-2}$ s$^{-1}$]")

# ax.legend();

# plot_path = Path(
#     "/home/parshap/cosipy/docs/tutorials/"
#     "spectral_fits/continuum_fit/AGN/NGC4151_CPL_fit.pdf"
# )
# fig.savefig(
#     plot_path,
#     bbox_inches="tight",
#     facecolor="white"
# )

In [ ]:
print("Starting fit...", flush=True)
_ = like.fit()
print("Fit complete.", flush=True)

like.results.display()

Now we can compare the fitted CPL with the injected DC3 EC1000 spectrum.

In [ ]:
results = like.results

parameters = {par.name: results.get_variates(par.path)
              for par in results.optimized_model["NGC_4151"].parameters.values()
              if par.free}

results_err = results.propagate(results.optimized_model["NGC_4151"].spectrum.main.shape.evaluate_at, **parameters)

In [ ]:
energy = np.geomspace(100*u.keV, 10*u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux = results_err(e)
    flux_median[i] = flux.median
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)
    flux_inj[i] = spectrum_inj.evaluate_at(e)

In [ ]:
%matplotlib inline

In [ ]:
fig, ax = plt.subplots(figsize = (9, 6))

ax.plot(energy, flux_median, label="Best-fit CPL")
ax.fill_between(energy, flux_lo, flux_hi, alpha=.35, label="Best-fit CPL (errors)")
ax.plot(energy, flux_inj, color="black", ls=":", lw=2,
        label="Injected CPL (cutoff = 1000 keV)")

ax.semilogx()
ax.semilogy()

ax.set_xlabel("Energy [keV]")
ax.set_ylabel(r"$\frac{\mathrm{d}N}{\mathrm{d}E}$ [keV$^{-1}$ cm$^{-2}$ s$^{-1}$]")

ax.legend()

plot_path = Path(
    "/home/parshap/cosipy/docs/tutorials/"
    "spectral_fits/continuum_fit/AGN/NGC4151_DC3_EC1000_CPL_fit.pdf"
)
fig.savefig(
    plot_path,
    bbox_inches="tight",
    facecolor="white"
)